# Single 3D Embedding View (Original vs Generated)

This notebook builds a single 3D embedding plot for one dataset and one generator type, using either PCA (3 components) or t-SNE (3 components).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Optional

from einops import rearrange

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import MinMaxScaler

import plotly.express as px
import plotly.io as pio

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
# --------------------------
# User config (edit here)
# --------------------------
DATASET_KEY = "UniMiB Running"  # UCB Normal, UCB Abnormal, UniMiB Running, UniMiB Jumping
GEN_TYPE = "type_1"         # type_1 or type_2
EMBED_METHOD = "pca"        # pca or tsne
NUM_SAMPLES = 1000
RANDOM_STATE = 43
SAVE_DIR = os.path.join(os.getcwd(), "outputs")
SAVE_FIG = False
SAVE_GIF = True          # set True to export a rotating 360-degree GIF

assert GEN_TYPE in ["type_1", "type_2"], "GEN_TYPE must be 'type_1' or 'type_2'"
assert EMBED_METHOD in ["pca", "tsne"], "EMBED_METHOD must be 'pca' or 'tsne'"

In [3]:
# Generator architecture copied from training/testing notebooks
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: Tensor, mask: Optional[Tensor] = None) -> Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum("bhqd, bhkd -> bhqk", queries, keys)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy = energy.masked_fill(~mask, fill_value)
        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum("bhal, bhlv -> bhav", att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out


class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )


class Gen_TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size, num_heads=5, drop_p=0.5, forward_expansion=4, forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p),
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p),
            )),
        )


class Gen_TransformerEncoder(nn.Sequential):
    def __init__(self, depth=8, **kwargs):
        super().__init__(*[Gen_TransformerEncoderBlock(**kwargs) for _ in range(depth)])


class Generator(nn.Module):
    def __init__(
        self,
        seq_len=150,
        patch_size=15,
        channels=3,
        latent_dim=100,
        embed_dim=10,
        depth=3,
        forward_drop_rate=0.5,
        attn_drop_rate=0.5,
    ):
        super().__init__()
        self.channels = channels
        self.latent_dim = latent_dim
        self.seq_len = seq_len
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.depth = depth
        self.attn_drop_rate = attn_drop_rate
        self.forward_drop_rate = forward_drop_rate

        self.l1 = nn.Linear(self.latent_dim, self.seq_len * self.embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.seq_len, self.embed_dim))
        self.blocks = Gen_TransformerEncoder(
            depth=self.depth,
            emb_size=self.embed_dim,
            drop_p=self.attn_drop_rate,
            forward_drop_p=self.forward_drop_rate,
        )
        self.deconv = nn.Sequential(nn.Conv2d(self.embed_dim, self.channels, 1, 1, 0))

    def forward(self, z):
        x = self.l1(z).view(-1, self.seq_len, self.embed_dim)
        x = x + self.pos_embed
        h, w = 1, self.seq_len
        x = self.blocks(x)
        x = x.reshape(x.shape[0], 1, x.shape[1], x.shape[2])
        output = self.deconv(x.permute(0, 3, 1, 2))
        output = output.view(-1, self.channels, h, w)
        return output

In [4]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATASET_SPECS = {
    "UCB Normal": {
        "folder": "test_ucb_normal",
        "kind": "ucb",
        "csv_path": os.path.join(ROOT, "consolidated_datasets", "pcb", "ptbdb_normal.csv"),
        "seq_len": 50,
        "channels": 1,
        "patch_size": 10,
        "latent_dim": 100,
        "use_scaler": False,
    },
    "UCB Abnormal": {
        "folder": "test_ucb_abnormal",
        "kind": "ucb",
        "csv_path": os.path.join(ROOT, "consolidated_datasets", "pcb", "ptbdb_abnormal.csv"),
        "seq_len": 50,
        "channels": 1,
        "patch_size": 10,
        "latent_dim": 100,
        "use_scaler": False,
    },
    "UniMiB Running": {
        "folder": "test_unimib_running",
        "kind": "unimib",
        "activity": "running",
        "data_type": "adl",
        "seq_len": 150,
        "channels": 3,
        "patch_size": 30,
        "latent_dim": 100,
        "use_scaler": True,
    },
    "UniMiB Jumping": {
        "folder": "test_unimib_jumping",
        "kind": "unimib",
        "activity": "jumping",
        "data_type": "adl",
        "seq_len": 150,
        "channels": 3,
        "patch_size": 30,
        "latent_dim": 100,
        "use_scaler": True,
    },
}


def _fit_normalizer_minus1_plus1(data_4d):
    scaler = MinMaxScaler(feature_range=(-1, 1))
    flat = data_4d.reshape(-1, data_4d.shape[-1])
    norm_flat = scaler.fit_transform(flat)
    norm_data = norm_flat.reshape(data_4d.shape)
    return norm_data, scaler


def _inverse_from_minus1_plus1(data_4d, scaler):
    flat = data_4d.reshape(-1, data_4d.shape[-1])
    inv_flat = scaler.inverse_transform(flat)
    return inv_flat.reshape(data_4d.shape)


def load_real_data(spec):
    import scipy.io

    if spec['kind'] == 'ucb':
        csv_path = spec['csv_path']
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Missing CSV file: {csv_path}")
        df = pd.read_csv(csv_path, header=None)
        seq_len = spec['seq_len']
        data = df.iloc[:, 5:5 + seq_len].to_numpy()
        data = data[:, np.newaxis, np.newaxis, :]
        return torch.tensor(data, dtype=torch.float32), None

    if spec['kind'] == 'unimib':
        data_type = spec['data_type']
        data_root = os.path.join(ROOT, 'consolidated_datasets', 'UniMiB-SHAR', 'data')
        data_mat = os.path.join(data_root, f"{data_type}_data.mat")
        labels_mat = os.path.join(data_root, f"{data_type}_labels.mat")
        names_mat = os.path.join(data_root, f"{data_type}_names.mat")

        if not (os.path.exists(data_mat) and os.path.exists(labels_mat) and os.path.exists(names_mat)):
            raise FileNotFoundError("One or more UniMiB .mat files are missing.")

        data_dict = scipy.io.loadmat(data_mat)
        labels_dict = scipy.io.loadmat(labels_mat)
        names_dict = scipy.io.loadmat(names_mat)

        acc_data = data_dict[f"{data_type}_data"]
        labels = labels_dict[f"{data_type}_labels"]
        activity_names = names_dict[f"{data_type}_names"]

        clean_activity_names = [name[0][0] for name in activity_names]
        target_activity = spec['activity']

        if target_activity not in clean_activity_names:
            raise ValueError(f"Activity '{target_activity}' not found in UniMiB names.")

        target_label = clean_activity_names.index(target_activity) + 1
        activity_ids = labels[:, 0]
        target_indices = np.where(activity_ids == target_label)[0]

        selected = acc_data[target_indices, :]
        selected = selected.reshape(-1, 3, 1, 151)[:, :, :, :spec['seq_len']]
        norm_selected, scaler = _fit_normalizer_minus1_plus1(selected)
        return torch.tensor(norm_selected, dtype=torch.float32), scaler

    raise ValueError(f"Unsupported dataset kind: {spec['kind']}")


def build_generator(spec, checkpoint_path, device='cpu'):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    model = Generator(
        seq_len=spec['seq_len'],
        patch_size=spec['patch_size'],
        channels=spec['channels'],
        latent_dim=spec['latent_dim'],
        depth=5,
    ).to(device)

    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def generate_samples(model, num_samples, latent_dim, device='cpu'):
    model.eval()
    with torch.no_grad():
        noise = torch.randn(num_samples, latent_dim, device=device)
        out = model(noise).detach().cpu().numpy()
    return out


def _to_seq_len_channels(batch_4d):
    return np.transpose(np.squeeze(batch_4d, axis=2), (0, 2, 1))


def _extract_rocket_features(fake_seq, real_seq):
    try:
        from sktime.transformations.panel.rocket import MiniRocketMultivariate
    except ImportError as exc:
        raise ImportError("MiniRocket requires sktime. Install with: pip install sktime") from exc

    dist = np.concatenate((fake_seq, real_seq), axis=0)
    dist = dist.transpose(0, 2, 1)
    rocket = MiniRocketMultivariate()
    features = rocket.fit_transform(dist)

    features_fake = features[:len(fake_seq)]
    features_real = features[len(fake_seq):]

    if hasattr(features_fake, 'to_numpy'):
        features_fake = features_fake.to_numpy()
    if hasattr(features_real, 'to_numpy'):
        features_real = features_real.to_numpy()

    return features_fake, features_real

In [5]:
def compute_3d_embedding(real_features, fake_features, method='pca', random_state=43):
    combined = np.concatenate([real_features, fake_features], axis=0)

    if method == 'pca':
        model = PCA(n_components=3)
        emb = model.fit_transform(combined)
        explained = model.explained_variance_ratio_
    elif method == 'tsne':
        n_total = combined.shape[0]
        max_valid = max(2, (n_total - 1) // 3)
        perplexity = min(30, max_valid)
        model = TSNE(
            n_components=3,
            random_state=random_state,
            init='pca',
            learning_rate='auto',
            perplexity=perplexity,
        )
        emb = model.fit_transform(combined)
        explained = None
    else:
        raise ValueError(f"Unsupported method: {method}")

    n_real = real_features.shape[0]
    emb_real = emb[:n_real]
    emb_fake = emb[n_real:]
    return emb_real, emb_fake, explained


def plot_single_3d_embedding(dataset_key, gen_type='type_1', method='pca', num_samples=1000, random_state=43, save_dir=None, save_fig=False):
    if dataset_key not in DATASET_SPECS:
        raise KeyError(f"Unknown dataset key: {dataset_key}")

    spec = DATASET_SPECS[dataset_key]
    folder = os.path.join(ROOT, spec['folder'])
    ckpt_subdir = 'ckpt_tmw_type_1' if gen_type == 'type_1' else 'ckpt_tmw_type_2'
    ckpt_path = os.path.join(folder, ckpt_subdir, 'generator_final.pth')

    real_data_model_space, scaler = load_real_data(spec)
    n_real = len(real_data_model_space)
    n_use = min(num_samples, n_real)
    if n_use < 10:
        raise ValueError(f"Dataset {dataset_key} has only {n_use} samples available.")

    real_idx = torch.randperm(n_real)[:n_use]
    real_batch_model_space = real_data_model_space[real_idx].numpy()

    model = build_generator(spec, ckpt_path, device=device)
    fake_model_space = generate_samples(model, n_use, spec['latent_dim'], device=device)

    if spec.get('use_scaler', False):
        real_batch = _inverse_from_minus1_plus1(real_batch_model_space, scaler)
        fake_batch = _inverse_from_minus1_plus1(fake_model_space, scaler)
    else:
        real_batch = real_batch_model_space
        fake_batch = fake_model_space

    real_seq = _to_seq_len_channels(real_batch)
    fake_seq = _to_seq_len_channels(fake_batch)
    fake_feat, real_feat = _extract_rocket_features(fake_seq, real_seq)

    emb_real, emb_fake, explained = compute_3d_embedding(
        real_features=real_feat,
        fake_features=fake_feat,
        method=method,
        random_state=random_state,
    )

    all_xyz = np.concatenate([emb_real, emb_fake], axis=0)
    labels = np.array(['Original'] * len(emb_real) + ['Generated'] * len(emb_fake))

    df_plot = pd.DataFrame({
        'Component 1': all_xyz[:, 0],
        'Component 2': all_xyz[:, 1],
        'Component 3': all_xyz[:, 2],
        'Source': labels,
    })

    method_upper = method.upper()
    title = f"{dataset_key} | {gen_type} | {method_upper} 3D"
    if method == 'pca' and explained is not None:
        var_text = f"Explained var: {explained[0]*100:.1f}% / {explained[1]*100:.1f}% / {explained[2]*100:.1f}%"
        title = title + "<br>" + var_text

    fig = px.scatter_3d(
        df_plot,
        x='Component 1',
        y='Component 2',
        z='Component 3',
        color='Source',
        color_discrete_map={'Original': 'royalblue', 'Generated': 'firebrick'},
        opacity=0.7,
        title=title,
    )

    fig.update_traces(marker=dict(size=4))
    fig.update_layout(
        template='plotly_white',
        legend_title_text='',
        margin=dict(l=0, r=0, b=0, t=70),
    )

    if save_fig:
        if save_dir is None:
            raise ValueError('save_dir must be provided when save_fig=True')
        os.makedirs(save_dir, exist_ok=True)
        out_name = f"single_3d_{dataset_key.replace(' ', '_').lower()}_{gen_type}_{method}.html"
        out_path = os.path.join(save_dir, out_name)
        pio.write_html(fig, out_path, auto_open=False, include_plotlyjs='cdn')
        print(f"Saved interactive HTML: {out_path}")

    fig.show()

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as _plt_gif
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation, PillowWriter


def save_rotating_gif(
    dataset_key,
    gen_type='type_1',
    method='pca',
    num_samples=1000,
    random_state=43,
    save_dir=None,
    n_frames=72,
    fps=20,
    elev=20,
):
    """Generate a 360-degree rotating GIF of the 3-D embedding scatter plot.

    Parameters
    ----------
    dataset_key : str
        One of the keys in DATASET_SPECS.
    gen_type : str
        'type_1' or 'type_2'.
    method : str
        'pca' or 'tsne'.
    num_samples : int
        Max number of samples to embed.
    random_state : int
        Random seed for reproducibility.
    save_dir : str
        Directory to write the .gif file.
    n_frames : int
        Number of animation frames (one full 360-degree rotation).
    fps : int
        Frames per second in the output GIF.
    elev : float
        Elevation angle (degrees) for the camera.
    """
    if dataset_key not in DATASET_SPECS:
        raise KeyError(f"Unknown dataset key: {dataset_key}")

    spec = DATASET_SPECS[dataset_key]
    folder = os.path.join(ROOT, spec['folder'])
    ckpt_subdir = 'ckpt_tmw_type_1' if gen_type == 'type_1' else 'ckpt_tmw_type_2'
    ckpt_path = os.path.join(folder, ckpt_subdir, 'generator_final.pth')

    # --- load & generate data (reuse helpers from above) ---
    real_data_model_space, scaler = load_real_data(spec)
    n_real = len(real_data_model_space)
    n_use = min(num_samples, n_real)
    if n_use < 10:
        raise ValueError(f"Dataset {dataset_key} has only {n_use} samples available.")

    rng = torch.Generator()
    rng.manual_seed(random_state)
    real_idx = torch.randperm(n_real, generator=rng)[:n_use]
    real_batch_model_space = real_data_model_space[real_idx].numpy()

    model = build_generator(spec, ckpt_path, device=device)
    fake_model_space = generate_samples(model, n_use, spec['latent_dim'], device=device)

    if spec.get('use_scaler', False):
        real_batch = _inverse_from_minus1_plus1(real_batch_model_space, scaler)
        fake_batch = _inverse_from_minus1_plus1(fake_model_space, scaler)
    else:
        real_batch = real_batch_model_space
        fake_batch = fake_model_space

    real_seq = _to_seq_len_channels(real_batch)
    fake_seq = _to_seq_len_channels(fake_batch)
    fake_feat, real_feat = _extract_rocket_features(fake_seq, real_seq)

    emb_real, emb_fake, explained = compute_3d_embedding(
        real_features=real_feat,
        fake_features=fake_feat,
        method=method,
        random_state=random_state,
    )

    # --- build matplotlib figure ---
    fig = _plt_gif.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(
        emb_real[:, 0], emb_real[:, 1], emb_real[:, 2],
        c='royalblue', alpha=0.55, s=8, label='Original',
    )
    ax.scatter(
        emb_fake[:, 0], emb_fake[:, 1], emb_fake[:, 2],
        c='firebrick', alpha=0.55, s=8, label='Generated',
    )

    method_upper = method.upper()
    title = f"{dataset_key} | {gen_type} | {method_upper} 3D"
    if method == 'pca' and explained is not None:
        var_text = (
            f"Var: {explained[0]*100:.1f}% / {explained[1]*100:.1f}% / {explained[2]*100:.1f}%"
        )
        title = title + '\n' + var_text
    ax.set_title(title, fontsize=10, pad=8)
    ax.set_xlabel(f'{method_upper} 1', fontsize=8)
    ax.set_ylabel(f'{method_upper} 2', fontsize=8)
    ax.set_zlabel(f'{method_upper} 3', fontsize=8)
    ax.legend(loc='upper left', fontsize=8)
    ax.tick_params(labelsize=7)
    fig.tight_layout()

    # --- rotation animation (360 degrees) ---
    azimuths = np.linspace(0, 360, n_frames, endpoint=False)

    def _update(frame):
        ax.view_init(elev=elev, azim=azimuths[frame])
        return (fig,)

    anim = FuncAnimation(fig, _update, frames=n_frames, interval=1000 // fps, blit=False)

    if save_dir is None:
        raise ValueError('save_dir must be provided')
    os.makedirs(save_dir, exist_ok=True)
    out_name = f"rotating_3d_{dataset_key.replace(' ', '_').lower()}_{gen_type}_{method}.gif"
    out_path = os.path.join(save_dir, out_name)
    anim.save(out_path, writer=PillowWriter(fps=fps))
    _plt_gif.close(fig)
    print(f"Saved rotating GIF: {out_path}")
    return out_path


In [7]:
plot_single_3d_embedding(
    dataset_key=DATASET_KEY,
    gen_type=GEN_TYPE,
    method=EMBED_METHOD,
    num_samples=NUM_SAMPLES,
    random_state=RANDOM_STATE,
    save_dir=SAVE_DIR,
    save_fig=SAVE_FIG,
)

In [8]:
if SAVE_GIF:
    save_rotating_gif(
        dataset_key=DATASET_KEY,
        gen_type=GEN_TYPE,
        method=EMBED_METHOD,
        num_samples=NUM_SAMPLES,
        random_state=RANDOM_STATE,
        save_dir=SAVE_DIR,
        n_frames=72,   # 72 frames = 5 degrees per frame
        fps=20,
        elev=20,
    )
else:
    print("SAVE_GIF is False — set it to True in the config cell to generate the rotating GIF.")


Saved rotating GIF: n:\Fun\TMW\image_generate\outputs\rotating_3d_unimib_running_type_1_pca.gif
